# Install thư viện

In [1]:
import subprocess
subprocess.run(["pip", "install", "azure-storage-blob", "-q"])

CompletedProcess(args=['pip', 'install', 'azure-storage-blob', '-q'], returncode=0)

 # Import

In [2]:
import requests
import json
import os
from datetime import datetime, timezone, timedelta
from azure.storage.blob import BlobServiceClient

# Xem format của NYC DOT

In [4]:
import requests

url = "https://linkdata.nyctmc.org/data/LinkSpeedQuery.txt"
response = requests.get(url, timeout=15)
lines = response.text.strip().split("\n")

header = lines[0].split("\t")
rows = [dict(zip(header, line.split("\t"))) for line in lines[1:]]

print("Header:", header)
print()
print("Row 0:", rows[0])

Header: ['"id"', '"Speed"', '"TravelTime"', '"Status"', '"DataAsOf"', '"linkId"', '"linkPoints"', '"EncodedPolyLine"', '"EncodedPolyLineLvls"', '"Owner"', '"Transcom_id"', '"Borough"', '"linkName"\r']

Row 0: {'"id"': '"1"', '"Speed"': '"0"', '"TravelTime"': '"0"', '"Status"': '"-101"', '"DataAsOf"': '"6/28/2026 09:06:14"', '"linkId"': '"4616337"', '"linkPoints"': '"40.74047,-74.009251 40.74137,-74.00893 40.7431706,-74.008591 40.7462304,-74.00797 40.74812,-74.007651 40.748701,-74.007691 40.74971,-74.00819 40.75048,-74.008321 40.751611,-74.00789 40.7537504,-74.00704 40.75721,-74.00463 40.76003,-74.002631 40.7607405,-7"', '"EncodedPolyLine"': '"}btwFx|ubMsD_AgJcAcR{ByJ_AsBFiEbByCXaFuAkLiDsTaNsPoKmCmB"', '"EncodedPolyLineLvls"': '"BBBBBBBBBBBBB"', '"Owner"': '"NYC_DOT_LIC"', '"Transcom_id"': '"4616337"', '"Borough"': '"Manhattan"', '"linkName"\r': '"11th ave n ganservoort - 12th ave @ 40th st"\r'}


# Gọi real-time feed và lọc 125 link active

In [5]:
import pytz

eastern = pytz.timezone("America/New_York")
now_et = datetime.now(eastern)
cutoff = now_et - timedelta(hours=48)

active_link_ids = []
for row in rows:
    try:
        data_as_of_str = row['"DataAsOf"'].strip().strip('"')
        link_id = row['"linkId"'].strip().strip('"')
        
        dt_naive = datetime.strptime(data_as_of_str, "%m/%d/%Y %H:%M:%S")
        dt_et = eastern.localize(dt_naive)
        
        if dt_et >= cutoff:
            active_link_ids.append(link_id)
    except Exception as e:
        continue

active_link_ids = sorted(set(active_link_ids))
print(f"Số link active: {len(active_link_ids)}")
print(active_link_ids[:5])

Số link active: 125
['4329472', '4329473', '4329499', '4329507', '4329508']


# Check lại TimeZone

In [6]:
print(f"Now ET:  {now_et}")
print(f"Cutoff:  {cutoff}")
print(f"DataAsOf mẫu: {dt_et}")

Now ET:  2026-06-28 09:05:37.689755-04:00
Cutoff:  2026-06-26 09:05:37.689755-04:00
DataAsOf mẫu: 2026-06-28 09:06:04-04:00


# Upload Azure

In [7]:
# Upload thẳng lên Azure
conn_str = os.environ["AZURE_STORAGE_CONNECTION_STRING"]
container = os.environ.get("AZURE_CONTAINER_NAME", "nyc-traffic-lakehouse")

client = BlobServiceClient.from_connection_string(conn_str)
blob = client.get_blob_client(container=container, blob="artifacts/active_link_ids.json")

data = json.dumps(active_link_ids).encode("utf-8")
blob.upload_blob(data, overwrite=True)

print("✅ Upload active_link_ids.json lên Azure thành công")

✅ Upload active_link_ids.json lên Azure thành công


# Verify

In [9]:
# Verify
downloaded = blob.download_blob().readall()
ids = json.loads(downloaded)
print(f"✅ Verify: {len(ids)} link trên Azure")
print(ids[:5])

✅ Verify: 125 link trên Azure
['4329472', '4329473', '4329499', '4329507', '4329508']
